# NuCLS E2 · Experimentos adicionales V1
Ejecutar **después de que terminen la cola actual y el cuaderno de cierre de reserva**. No iniciar en paralelo. E1 y los checkpoints originales se leen, no se modifican.

Este cuaderno realiza: ensembles de los tres mejores seleccionados por validación, incertidumbre y errores; control de sanidad; Grad-CAM y borrado contra control aleatorio; tres ablaciones con cinco semillas cada una. Reutiliza los controles existentes y reanuda las ablaciones desde la última época guardada. Una época interrumpida puede repetirse; no se promete reanudación por lote.

**PCam es opcional y costoso:** cinco entrenamientos de 20 épocas con 262.144 imágenes; no cabe inferir su duración a partir de NuCLS. Permanece desactivado para priorizar el cierre urgente. Su implementación es una réplica parcial con diferencias técnicas declaradas, pendiente de prueba GPU. Los CSV de errores y ruido requieren revisión humana. Código probado localmente; no es un notebook ya ejecutado en GPU.


In [ ]:
from pathlib import Path
import os,sys,json,subprocess,zipfile,io,base64,hashlib
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
from google.colab import drive
drive.mount('/content/drive')
DRIVE=Path('/content/drive/MyDrive')
EJECUTAR_PCAM=False
CERRADO=DRIVE/'NuCLS_E2_CIERRE_RESERVA_V1'
for folder in ['NuCLS_E2_ADICIONALES_EXPLORATORIO','NuCLS_E2_TRES_NUEVOS_V1','NuCLS_E2_COLA_UNICA_V1','NuCLS_E2_CIERRE_RESERVA_V1','NuCLS_E2_EXPERIMENTOS_POST_V1','NuCLS_E2_REPLICA_PCAM_V1']:
    if (DRIVE/folder/'EJECUCION_ACTIVA.json').exists():
        raise RuntimeError('Hay una marca de ejecución: '+folder+'. No se modificó. Esperar a que termine y verificar la sesión.')
assert (CERRADO/'estado.json').is_file(), 'Falta ejecutar el cuaderno de cierre en esta cuenta de Drive.'
assert json.loads((CERRADO/'estado.json').read_text())['estado']=='CIERRE_RESERVA_Y_ESTADISTICA_COMPLETADO', 'El cierre de reserva aún no está completo.'
assert (DRIVE/'NuCLS_V2_DATOS.zip').is_file(), 'Falta el ZIP original de datos en esta cuenta.'
print('Fuentes encontradas. No se inició entrenamiento.')


In [ ]:
raw=base64.b64decode('...')
assert hashlib.sha256(raw).hexdigest()=='0850839ac32a6d7fcce3e9608529fc9d7e5f9b483a6a4ba25d6dc1314c4bdd78'
CODIGO=Path('/content/nucls_e2_post_v1_codigo');CODIGO.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    for name in z.namelist():
        target=(CODIGO/name).resolve()
        assert target.is_relative_to(CODIGO.resolve())
        target.write_bytes(z.read(name))
subprocess.run([sys.executable,'-m','pip','install','timm==1.0.22','h5py'],check=True)
subprocess.run([sys.executable,str(CODIGO/'test_transformaciones.py')],check=True)
subprocess.run([sys.executable,str(CODIGO/'test_estadistica.py')],check=True)
subprocess.run([sys.executable,str(CODIGO/'test_reanudacion.py')],check=True)
print('Código verificado y pruebas aprobadas.')


In [ ]:
import datetime
SALIDA=DRIVE/'NuCLS_E2_EXPERIMENTOS_POST_V1';SALIDA.mkdir(exist_ok=True)
log=SALIDA/('log_'+datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'.txt')
def ejecutar(script,*args):
    command=[sys.executable,'-u',str(CODIGO/script),*map(str,args)]
    with log.open('a') as f:
        f.write('\n'+repr(command)+'\n');f.flush()
        proceso=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
        try:
            for line in proceso.stdout:
                print(line,end='',flush=True);f.write(line);f.flush()
            retorno=proceso.wait()
        except BaseException:
            proceso.terminate();proceso.wait();raise
    if retorno:raise RuntimeError(f'{script} se detuvo con código {retorno}. Se conservó lo guardado. Ver '+str(log))
ejecutar('ensembles.py','--cierre',CERRADO,'--salida',SALIDA/'ensembles')
ejecutar('experimentos.py','--drive',DRIVE)
if EJECUTAR_PCAM:
    ejecutar('replica_pcam.py','--datos','/content/pcam_oficial','--salida',DRIVE/'NuCLS_E2_REPLICA_PCAM_V1')
print('ADICIONALES_SOLICITADOS_COMPLETADOS')
print('Resultados:', SALIDA.parent/(SALIDA.name+'_RESULTADOS.zip'))
print('Guardar este notebook con salidas: Archivo → Descargar → Descargar .ipynb')


## Alcance y entrega
El ZIP de adicionales excluye pesos, que quedan en Drive. Si se habilitó PCam, sus resultados quedan separados en `NuCLS_E2_REPLICA_PCAM_V1`; conservar también esa carpeta y sus checkpoints. Los indicadores de sanidad fallida o Grad-CAM no aplicable no se convierten en aprobación automáticamente.

Los tres mejores se seleccionan únicamente con validación. Las ablaciones no vuelven a abrir la reserva. La réplica PCam usa su propia prueba oficial una vez congelados sus cinco modelos. El techo multi-rater se calcula aparte con `techo_multirater.py` cuando existan anotaciones verificables; no se inventa una cota.
